# Stage 2 synthesis CV (Section 5.3)

Act IV synthesis CV (`StratifiedGroupKFold` on `experimental_group`, 10-fold default).

**Stage 1:** fit **once** per training scenario (A/B/C) on the official train/test split (§5.2); pooled test metrics in logs mix cohorts — use the **Stage 1 classification table** cell for Brad vs Liberman columns. **Stage 2** only is refit per CV fold.

**Artifacts (separate by fold count):**
- **10-fold:** `figures/cache/stage2_synthesis_cv_{folds,progress,summary}.*` (legacy names)
- **5-fold:** `figures/cache/stage2_synthesis_cv_5fold_*`

Set `FORCE_RERUN` per run below. Completed keys are skipped within each fold-count cache.


In [1]:
import importlib
import json
from pathlib import Path

import pandas as pd

import utils.stage2_synthesis_cv as _synthesis_cv
import utils.stage2_synthesis_plot as _synthesis_plot

importlib.reload(_synthesis_cv)
importlib.reload(_synthesis_plot)

from utils.benchmark_metrics import (
    LIBERMAN_GROUP_MEAN_BASELINE_PARQUET,
    STAGE2_BEST_HP_DIR,
    STAGE2_SYNTHESIS_CV_OOF_PARQUET,
    STAGE2_SYNTHESIS_CV_OOF_R2_SUMMARY_PARQUET,
    STAGE2_SYNTHESIS_CV_POOLED_OOF_PARQUET,
    STAGE2_SYNTHESIS_STAGE1_TABLE_PARQUET,
    stage2_synthesis_cv_paths,
)
from utils.nn_stage2_data import load_nn_stage2_data
from utils.stage2_hp import resolve_torch_device
from utils.stage2_synthesis_cv import (
    build_stage1_classification_table,
    compute_pooled_synthesis_folds,
    panel_c_rmse_report_table,
    pooled_oof_r2_report_table,
    run_synthesis_cv,
    summarize_pooled_folds,
    summarize_pooled_oof_r2,
)
from utils.stage2_synthesis_plot import (
    synthesis_act4_grid_cv,
    synthesis_cohort_panels_3cv,
    synthesis_cohort_panels_3cv_r2,
    synthesis_cohort_panels_cv,
)
from utils.subject_cv import eval_stratum_mean_baseline_cv

device = resolve_torch_device()
print("device:", device)

device: mps


In [2]:
# Stage 1 classification: per-cohort + pooled overall test acc/AUC (33 animals)
stage1_table = build_stage1_classification_table(verbose=True)
stage1_table.to_parquet(STAGE2_SYNTHESIS_STAGE1_TABLE_PARQUET, index=False)
display(
    stage1_table[
        [
            "scenario",
            "training_data",
            "brad_test_acc",
            "brad_test_auc",
            "liberman_test_acc",
            "liberman_test_auc",
            "overall_test_acc",
            "overall_test_auc",
        ]
    ].round(3)
)


=== Stage 1 table: within-cohort (Brad) ===
           Selected RF  — fit acc (animal@cal): 0.944 | non-test acc (animal@cal): 0.933 | threshold (Fit+Val): 0.423 | test acc (animal@cal): 0.917 | test acc@0.5: 0.583 | AUC: 0.944

=== Stage 1 table: within-cohort (Liberman) ===
           Selected RF  — fit acc (animal@cal): 0.971 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal@cal): 0.952 | test acc@0.5: 0.810 | AUC: 0.909

=== Stage 1 table: cross-cohort (train Brad → test Liberman) ===
           Selected RF  — fit acc (animal@cal): 0.944 | non-test acc (animal@cal): 0.933 | threshold (Fit+Val): 0.423 | test acc (animal@cal): 0.619 | test acc@0.5: 0.905 | AUC: 0.936

=== Stage 1 table: cross-cohort (train Liberman → test Brad) ===
           Selected RF  — fit acc (animal@cal): 0.971 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal@cal): 0.500 | test acc@0.5: 0.500 | AUC: 0.194

=== Stage 1 table: combined training 

,scenario,training_data,brad_test_acc,brad_test_auc,liberman_test_acc,liberman_test_auc
0,1,Within-cohort,0.917,0.944,0.952,0.909
1,2,Cross-cohort,0.500,0.194,0.619,0.936
2,3,Combined,0.833,0.722,0.762,0.918


In [3]:
hp = {}
for scen in ("A", "B", "C"):
    p = STAGE2_BEST_HP_DIR / f"{scen}.json"
    hp[scen] = json.loads(p.read_text())

# (n_splits, force_rerun) — separate cache files per fold count
# After feature changes (e.g. strain_binary), delete stage2_synthesis_cv_* parquets
# or set force_rerun=True for that run entry.
SYNTHESIS_RUNS = [
    (10, True),  # legacy paths; skip if already complete
    (5, True),  # figures/cache/stage2_synthesis_cv_5fold_*
]
summaries = {}
for n_folds, force_rerun in SYNTHESIS_RUNS:
    _, summary = run_synthesis_cv(
        hp,
        device=device,
        n_splits=n_folds,
        force_rerun=force_rerun,
        verbose=True,
    )
    summaries[n_folds] = summary
    print(stage2_synthesis_cv_paths(n_folds)[2])

summary_10 = summaries[10]
summary_5 = summaries[5]
summary_10

Synthesis CV: 10-fold → stage2_synthesis_cv_summary.parquet

=== Global Stage 1 (per training scenario A/B/C) ===
           Selected RF  — fit acc (animal@cal): 0.944 | non-test acc (animal@cal): 0.933 | threshold (Fit+Val): 0.423 | test acc (animal@cal): 0.917 | test acc@0.5: 0.583 | AUC: 0.944
  global S1 scen=A (RF): fit/val 45 animals, official test 12 animals, labels for 162 animals | Brad test acc=0.917 AUC=0.944 | Lib test acc=nan AUC=nan
           Selected RF  — fit acc (animal@cal): 0.865 | non-test acc (animal@cal): 0.860 | threshold (Fit+Val): 0.458 | test acc (animal@cal): 0.788 | test acc@0.5: 0.727 | AUC: 0.805
  global S1 scen=B (RF): fit/val 129 animals, official test 33 animals, labels for 162 animals | Brad test acc=0.833 AUC=0.722 | Lib test acc=0.762 AUC=0.918
           Selected RF  — fit acc (animal@cal): 0.971 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal@cal): 0.952 | test acc@0.5: 0.810 | AUC: 0.909
  global S1 scen=C (RF)

,test_set,scenario,model,RMSE,RMSE_sem,source,n_folds
0,Brad,all,ceiling,2.961628,0.149565,ceiling,10
1,Liberman,all,ceiling,2.341969,0.098753,ceiling,10
2,Brad,A,LR baseline,3.284766,0.114825,classical,10
3,Brad,A,RF,2.795391,0.137717,classical,10
4,Brad,A,XGB,2.728043,0.145813,classical,10
5,Brad,A,MLP,2.386861,0.165069,nn,10
6,Brad,B,LR baseline,3.673654,0.182984,classical,10
7,Brad,B,RF,2.945283,0.144924,classical,10
8,Brad,B,XGB,2.836683,0.157263,classical,10
9,Brad,B,MLP,2.586836,0.169115,nn,10


In [4]:
summary_5

,test_set,scenario,model,RMSE,RMSE_sem,source,n_folds
0,Brad,all,ceiling,2.925654,0.129869,ceiling,5
1,Liberman,all,ceiling,2.314670,0.117805,ceiling,5
2,Brad,A,LR baseline,3.270088,0.075996,classical,5
3,Brad,A,RF,2.831451,0.074894,classical,5
4,Brad,A,XGB,2.823922,0.078911,classical,5
5,Brad,A,MLP,2.393371,0.162426,nn,5
6,Brad,B,LR baseline,3.717164,0.140497,classical,5
7,Brad,B,RF,2.986284,0.101157,classical,5
8,Brad,B,XGB,2.836939,0.139167,classical,5
9,Brad,B,MLP,2.550488,0.158097,nn,5


In [5]:
# Ceiling sanity: Liberman vs cached baseline; Brad inline recompute
STRATUM_COLS = ["noise_cat", "frequency"]
data = load_nn_stage2_data()
lib_cv = eval_stratum_mean_baseline_cv(
    data.reformatted_orig.reset_index(drop=True), STRATUM_COLS
)
lib_ref = pd.read_parquet(LIBERMAN_GROUP_MEAN_BASELINE_PARQUET)
print("Liberman CV rmse", lib_cv["rmse"], "±", lib_cv["rmse_sem"])
print(
    "Liberman parquet rmse",
    float(lib_ref["rmse"].iloc[0]),
    "±",
    float(lib_ref["rmse_sem"].iloc[0]),
)
brad_cv = eval_stratum_mean_baseline_cv(
    data.reformatted.reset_index(drop=True), STRATUM_COLS
)
print("Brad inline CV rmse", brad_cv["rmse"], "±", brad_cv["rmse_sem"])
for n_folds, summary in summaries.items():
    print(f"\n=== {n_folds}-fold ceiling (from synthesis summary) ===")
    print(summary.loc[summary["model"].eq("ceiling")])

Liberman CV rmse 2.3419685439445384 ± 0.09875276799742796
Liberman parquet rmse 2.3419685439445384 ± 0.09875276799742796
Brad inline CV rmse 2.9616282300167436 ± 0.14956470965710605

=== 10-fold ceiling (from synthesis summary) ===
   test_set scenario    model      RMSE  RMSE_sem   source  n_folds
0      Brad      all  ceiling  2.961628  0.149565  ceiling       10
1  Liberman      all  ceiling  2.341969  0.098753  ceiling       10

=== 5-fold ceiling (from synthesis summary) ===
   test_set scenario    model      RMSE  RMSE_sem   source  n_folds
0      Brad      all  ceiling  2.925654  0.129869  ceiling        5
1  Liberman      all  ceiling  2.314670  0.117805  ceiling        5


In [6]:
for n_folds, summary in summaries.items():
    png = synthesis_act4_grid_cv(summary, n_folds=n_folds)
    print(f"{n_folds}-fold saved", png)

10-fold saved /Users/nowaki027/MSDS/Practicum/figures/presentation/deck_act4_synthesis_grid_RMSE_cv.png
5-fold saved /Users/nowaki027/MSDS/Practicum/figures/presentation/deck_act4_synthesis_grid_RMSE_cv_5fold.png


## Panel C — pooled holdout RMSE (both cohorts per fold)

Recomputes animal×frequency RMSE by concatenating Brad + Liberman eval aggs per fold (not the mean of cohort fold RMSEs). Pooled rows use **`scenario` 1 / 2 / 3** = within-cohort / cross-cohort / combined (distinct from per-cohort train-slice **A / B / C**). Review the table before running the 3-panel figure below.

In [8]:
FORCE_POOLED_RERUN = True
pooled_folds = compute_pooled_synthesis_folds(
    hp, device=device, verbose=True, force_rerun=FORCE_POOLED_RERUN
)
summary_pooled = summarize_pooled_folds(pooled_folds)
display(panel_c_rmse_report_table(summary_pooled))
summary_10_pooled = pd.concat([summary_10, summary_pooled], ignore_index=True)
display(summary_10_pooled.loc[summary_10_pooled["test_set"].eq("Pooled")])

Pooled synthesis CV: 10-fold → stage2_synthesis_cv_pooled_folds.parquet
pooled fold 0 ceiling rmse=2.5794
pooled f0 display=1 L7 rmse=3.0256
pooled f0 display=1 RF rmse=2.3231
pooled f0 display=1 XGB rmse=2.1444
pooled f0 display=1 MLP rmse=1.9836
pooled f0 display=2 L7 rmse=4.1174
pooled f0 display=2 RF rmse=4.6359
pooled f0 display=2 XGB rmse=4.6415
pooled f0 display=2 MLP rmse=4.2921
pooled f0 display=3 L7 rmse=3.2763
pooled f0 display=3 RF rmse=2.6244
pooled f0 display=3 XGB rmse=2.4497
pooled f0 display=3 MLP rmse=1.8790
pooled fold 1 ceiling rmse=2.6754
pooled f1 display=1 L7 rmse=3.0518
pooled f1 display=1 RF rmse=2.4816
pooled f1 display=1 XGB rmse=2.3030
pooled f1 display=1 MLP rmse=2.3499
pooled f1 display=2 L7 rmse=4.2034
pooled f1 display=2 RF rmse=4.8577
pooled f1 display=2 XGB rmse=4.9675
pooled f1 display=2 MLP rmse=4.0887
pooled f1 display=3 L7 rmse=3.4468
pooled f1 display=3 RF rmse=2.7655
pooled f1 display=3 XGB rmse=2.5694
pooled f1 display=3 MLP rmse=2.0671
pooled f

,model,Within-cohort,Cross-cohort,Combined
0,LR baseline,3.1698 ± 0.1134,4.1205 ± 0.0875,3.4419 ± 0.0895
1,RF,2.5497 ± 0.1089,4.6025 ± 0.1187,2.8064 ± 0.1098
2,XGB,2.5093 ± 0.1176,4.7267 ± 0.1082,2.6833 ± 0.0974
3,MLP,2.3742 ± 0.1108,4.1272 ± 0.0875,2.4704 ± 0.1023
4,ceiling (stratum mean),,,2.7558 ± 0.0679


,test_set,scenario,model,RMSE,RMSE_sem,source,n_folds
26,Pooled,all,ceiling,2.755841,0.067898,ceiling,10
27,Pooled,A,LR baseline,3.169794,0.113435,classical,10
28,Pooled,A,RF,2.549694,0.108926,classical,10
29,Pooled,A,XGB,2.509282,0.117596,classical,10
30,Pooled,A,MLP,2.374202,0.110806,nn,10
31,Pooled,B,LR baseline,4.120477,0.087476,classical,10
32,Pooled,B,RF,4.602473,0.118718,classical,10
33,Pooled,B,XGB,4.726730,0.108156,classical,10
34,Pooled,B,MLP,4.127185,0.087519,nn,10
35,Pooled,C,LR baseline,3.441903,0.089532,classical,10


In [9]:
png = synthesis_cohort_panels_cv(summary_10)
svg = png.with_suffix(".svg")
print("10-fold cohort panels saved", png, "+", svg)

10-fold cohort panels saved /Users/nowaki027/MSDS/Practicum/figures/presentation/deck_act4_synthesis_cohorts_RMSE_cv.png + /Users/nowaki027/MSDS/Practicum/figures/presentation/deck_act4_synthesis_cohorts_RMSE_cv.svg


In [10]:
png3 = synthesis_cohort_panels_3cv(summary_10_pooled)
svg3 = png3.with_suffix(".svg")
print("10-fold 3-panel cohort figure saved", png3, "+", svg3)

10-fold 3-panel cohort figure saved /Users/nowaki027/MSDS/Practicum/figures/presentation/deck_act4_synthesis_cohorts_RMSE_cv_3panel.png + /Users/nowaki027/MSDS/Practicum/figures/presentation/deck_act4_synthesis_cohorts_RMSE_cv_3panel.svg


## §5.2.2 — Pooled out-of-fold R² + Supplementary Figure S2

Backfills OOF prediction caches (separate progress from RMSE). Pooled OOF uses the same `compute_pooled_synthesis_folds` protocol as Panel C (not a Liberman+Brad concat).

In [11]:
# OOF backfill: reuses RMSE cache keys; only missing OOF rows are recomputed.
FORCE_OOF_RERUN = False
if FORCE_OOF_RERUN:
    from utils.stage2_synthesis_cv import (
        reset_pooled_oof_cache,
        reset_synthesis_oof_cache,
    )

    reset_synthesis_oof_cache()
    reset_pooled_oof_cache()

_, _ = run_synthesis_cv(hp, device=device, verbose=True, force_rerun=False)
pooled_folds = compute_pooled_synthesis_folds(
    hp, device=device, verbose=True, force_rerun=False
)

cohort_oof = pd.read_parquet(STAGE2_SYNTHESIS_CV_OOF_PARQUET)
pooled_oof = pd.read_parquet(STAGE2_SYNTHESIS_CV_POOLED_OOF_PARQUET)
print(
    "cohort OOF rows:", len(cohort_oof), "| pooled OOF rows:", len(pooled_oof)
)

r2_summary_fig = summarize_pooled_oof_r2(cohort_oof, pooled_oof)
display(pooled_oof_r2_report_table(r2_summary_fig))
r2_summary_fig.to_parquet(
    STAGE2_SYNTHESIS_CV_OOF_R2_SUMMARY_PARQUET, index=False
)
print("saved", STAGE2_SYNTHESIS_CV_OOF_R2_SUMMARY_PARQUET)

Synthesis CV: 10-fold → stage2_synthesis_cv_summary.parquet

=== Global Stage 1 (per training scenario A/B/C) ===
           Selected RF  — fit acc (animal@cal): 0.944 | non-test acc (animal@cal): 0.933 | threshold (Fit+Val): 0.423 | test acc (animal@cal): 0.917 | test acc@0.5: 0.583 | AUC: 0.944
  global S1 scen=A (RF): fit/val 45 animals, official test 12 animals, labels for 162 animals | Brad test acc=0.917 AUC=0.944 | Lib test acc=nan AUC=nan
           Selected RF  — fit acc (animal@cal): 0.865 | non-test acc (animal@cal): 0.860 | threshold (Fit+Val): 0.458 | test acc (animal@cal): 0.788 | test acc@0.5: 0.727 | AUC: 0.805
  global S1 scen=B (RF): fit/val 129 animals, official test 33 animals, labels for 162 animals | Brad test acc=0.833 AUC=0.722 | Lib test acc=0.762 AUC=0.918
           Selected RF  — fit acc (animal@cal): 0.971 | non-test acc (animal@cal): 0.940 | threshold (Fit+Val): 0.556 | test acc (animal@cal): 0.952 | test acc@0.5: 0.810 | AUC: 0.909
  global S1 scen=C (RF)

,model,Cohort A (Liberman),Cohort B (Brad),Pooled
0,LR baseline,Within-cohort: -0.4769; Cross-cohort: 0.0069; ...,Within-cohort: 0.3923; Cross-cohort: 0.2234; C...,Within-cohort: 0.3216; Cross-cohort: -0.1352; ...
1,RF,Within-cohort: -0.6384; Cross-cohort: 0.3194; ...,Within-cohort: 0.5498; Cross-cohort: 0.4991; C...,Within-cohort: 0.5585; Cross-cohort: -0.4195; ...
2,XGB,Within-cohort: -0.7546; Cross-cohort: 0.3857; ...,Within-cohort: 0.5610; Cross-cohort: 0.5396; C...,Within-cohort: 0.5707; Cross-cohort: -0.4946; ...
3,MLP,Within-cohort: -0.2061; Cross-cohort: 0.4979; ...,Within-cohort: 0.6697; Cross-cohort: 0.6251; C...,Within-cohort: 0.6278; Cross-cohort: -0.1045; ...
4,ceiling (stratum mean),0.5041 ± 0.0540,0.5091 ± 0.0381,0.4920 ± 0.0201


saved /Users/nowaki027/MSDS/Practicum/figures/cache/stage2_synthesis_cv_oof_r2_summary.parquet


In [12]:
png_s2 = synthesis_cohort_panels_3cv_r2(r2_summary_fig)
svg_s2 = png_s2.with_suffix(".svg")
print("Supplementary Figure S2 saved", png_s2, "+", svg_s2)

Supplementary Figure S2 saved /Users/nowaki027/MSDS/Practicum/figures/presentation/supplementary_figure_S2_pooled_oof_r2.png + /Users/nowaki027/MSDS/Practicum/figures/presentation/supplementary_figure_S2_pooled_oof_r2.svg


In [13]:
# 5-fold vs 10-fold RMSE (Liberman scenario C highlighted)
key = ["test_set", "scenario", "model"]
cmp = summary_10.merge(summary_5, on=key, suffixes=("_10", "_5"))
cmp["RMSE_delta_5_minus_10"] = cmp["RMSE_5"] - cmp["RMSE_10"]
cols = key + [
    "RMSE_10",
    "RMSE_sem_10",
    "RMSE_5",
    "RMSE_sem_5",
    "RMSE_delta_5_minus_10",
]
display(cmp[cols].sort_values(["test_set", "scenario", "model"]))

lib_c = cmp.loc[
    cmp["test_set"].eq("Liberman")
    & cmp["scenario"].eq("C")
    & cmp["model"].eq("RF")
]
if len(lib_c):
    r = lib_c.iloc[0]
    print(
        f"Liberman C RF: 10-fold {r.RMSE_10:.3f} ± {r.RMSE_sem_10:.3f} | "
        f"5-fold {r.RMSE_5:.3f} ± {r.RMSE_sem_5:.3f}"
    )

,test_set,scenario,model,RMSE_10,RMSE_sem_10,RMSE_5,RMSE_sem_5,RMSE_delta_5_minus_10
2,Brad,A,LR baseline,3.284766,0.114825,3.270088,0.075996,-0.014678
5,Brad,A,MLP,2.386861,0.165069,2.393371,0.162426,0.006509
3,Brad,A,RF,2.795391,0.137717,2.831451,0.074894,0.036060
4,Brad,A,XGB,2.728043,0.145813,2.823922,0.078911,0.095879
6,Brad,B,LR baseline,3.673654,0.182984,3.717164,0.140497,0.043510
9,Brad,B,MLP,2.586836,0.169115,2.550488,0.158097,-0.036347
7,Brad,B,RF,2.945283,0.144924,2.986284,0.101157,0.041001
8,Brad,B,XGB,2.836683,0.157263,2.836939,0.139167,0.000256
10,Brad,C,LR baseline,4.243655,0.210388,4.263235,0.146083,0.019581
13,Brad,C,MLP,4.604082,0.241428,4.709754,0.215452,0.105672


Liberman C RF: 10-fold 2.380 ± 0.140 | 5-fold 2.380 ± 0.118


In [14]:
# Act IV-style digest (mean ± SEM)
for n_folds, summary in summaries.items():
    print(f"\n######## {n_folds}-fold ########")
    for ts in ("Brad", "Liberman"):
        for scen in ("A", "B", "C"):
            sub = summary.loc[
                summary["test_set"].eq(ts) & summary["scenario"].eq(scen)
            ]
            print(f"\n=== {ts} test | train {scen} ===")
            print(
                sub[["model", "RMSE", "RMSE_sem", "source"]].to_string(
                    index=False
                )
            )


######## 10-fold ########

=== Brad test | train A ===
      model     RMSE  RMSE_sem    source
LR baseline 3.284766  0.114825 classical
         RF 2.795391  0.137717 classical
        XGB 2.728043  0.145813 classical
        MLP 2.386861  0.165069        nn

=== Brad test | train B ===
      model     RMSE  RMSE_sem    source
LR baseline 3.673654  0.182984 classical
         RF 2.945283  0.144924 classical
        XGB 2.836683  0.157263 classical
        MLP 2.586836  0.169115        nn

=== Brad test | train C ===
      model     RMSE  RMSE_sem    source
LR baseline 4.243655  0.210388 classical
         RF 5.138148  0.254856 classical
        XGB 5.224184  0.255678 classical
        MLP 4.604082  0.241428        nn

=== Liberman test | train A ===
      model     RMSE  RMSE_sem    source
LR baseline 4.037380  0.097651 classical
         RF 4.237575  0.170997 classical
        XGB 4.394067  0.148616 classical
        MLP 3.689947  0.165689        nn

=== Liberman test | train B ===


In [ ]:
import importlib
import utils.stage2_synthesis_plot as _sp

importlib.reload(_sp)
from utils.stage2_synthesis_plot import synthesis_cohort_panels_3cv

png3 = synthesis_cohort_panels_3cv(summary_10_pooled)